In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

In [50]:
import re

In [34]:
# --- 1. SETUP ---
# Define the path to the folder containing your 48 CSV files in Google Drive.
folder_path = '/content/drive/MyDrive/GEE_Exports'

# Define the name and path for the final combined output file.
output_file = '/content/drive/MyDrive/GEE_Exports/md_soybean_ndvi_timeseries_combined_wide.csv'

In [44]:
# --- 2. GET THE LIST OF CSV FILES ---
try:
    all_files = os.listdir(folder_path)

    # MODIFIED: Looks for the new filenames you created.
    csv_files = [f for f in all_files if f.startswith('md_county_soybean_ndvi_') and f.endswith('.csv')]

    print(f"Found {len(csv_files)} soybean-specific CSV files to process.")
except FileNotFoundError:
    print(f"Error: The folder '{folder_path}' was not found.")
    csv_files = []

Found 48 soybean-specific CSV files to process.


In [51]:
# --- 3. LOOP, READ, AND COMBINE (WITH ROBUST DATE PARSING) ---
if csv_files:
    all_data_list = []

    month_map = {
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'may': '05', 'jun': '06',
        'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    for filename in csv_files:
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # --- THIS IS THE FIX: More robust way to find month and year ---
        # Extracts any 3-letter month followed by a year from the filename
        match = re.search(r'([a-zA-Z]{3,4})_?(\d{4})', filename)

        if match:
            month_abbr = match.group(1).lower()
            year = match.group(2)

            if month_abbr in month_map:
                date_str = f"{year}-{month_map[month_abbr]}"
                df['date'] = date_str
                # Ensure 'date' column exists before appending
                if 'date' in df.columns:
                    all_data_list.append(df)
                else:
                    print(f"WARNING: 'date' column not created for {filename}. Skipping.")
            else:
                print(f"WARNING: Found month '{month_abbr}' but it's not in the map. Skipping {filename}")
        else:
            print(f"WARNING: Could not find a date pattern in filename: {filename}")

    # --- The rest of the script is the same ---

    if not all_data_list:
        print("\nERROR: No data was collected. Please check filenames and parsing logic.")
    else:
        long_df = pd.concat(all_data_list, ignore_index=True)
        print(f"\nSuccessfully created long_df. Shape: {long_df.shape}")
        print("Unique dates found:", sorted(long_df['date'].unique()))


        # --- 4. PIVOT THE LONG DATAFRAME TO THE WIDE FORMAT ---
        print("\nPivoting the table to a wide format...")
        wide_df = long_df.pivot_table(index='NAME', columns='date', values='meanSoybeanNDVI', aggfunc='mean')

        wide_df = wide_df.sort_index(axis=1)
        wide_df.columns = ['SoybeanNDVI_' + str(col) for col in wide_df.columns]

        print("Pivoting complete. Final table shape:", wide_df.shape)
        print("Here is a preview of your combined table:")
        display(wide_df.head())

        # --- 5. SAVE THE FINAL WIDE DATAFRAME TO A NEW CSV ---
        wide_df.to_csv(output_file)
        print(f"\n✅ Success! The combined soybean data has been saved to: '{output_file}'")


Successfully created long_df. Shape: (2491, 3)
Unique dates found: ['2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12', '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

Pivoting the table to a wide format...
Pivoting complete. Final table shape: (50, 47)
Here is a preview of your combined table:


,SoybeanNDVI_2021-01,SoybeanNDVI_2021-02,SoybeanNDVI_2021-03,SoybeanNDVI_2021-04,SoybeanNDVI_2021-05,SoybeanNDVI_2021-06,SoybeanNDVI_2021-07,SoybeanNDVI_2021-08,SoybeanNDVI_2021-09,SoybeanNDVI_2021-10,...,SoybeanNDVI_2024-03,SoybeanNDVI_2024-04,SoybeanNDVI_2024-05,SoybeanNDVI_2024-06,SoybeanNDVI_2024-07,SoybeanNDVI_2024-08,SoybeanNDVI_2024-09,SoybeanNDVI_2024-10,SoybeanNDVI_2024-11,SoybeanNDVI_2024-12
NAME,,,,,,,,,,,,,,,,,,,,,
Accomack,0.445960,0.435757,0.417194,0.548986,0.469426,0.414211,0.516794,0.773962,0.793315,0.520370,...,0.567101,0.566077,0.374402,0.290968,0.548027,0.860622,0.834187,0.522341,0.328489,0.346335
Adams,0.418706,0.593133,0.400964,0.514504,0.486666,0.428843,0.651924,0.750870,0.822280,0.451211,...,0.438499,0.529359,0.421157,0.292993,0.495352,0.824034,0.809289,0.411620,0.329359,0.350501
Alexandria,0.313524,0.302045,0.295838,0.432692,0.527326,0.556720,0.509160,0.524693,0.522667,0.496570,...,0.309551,0.344335,0.452578,0.460169,0.587002,0.742844,0.761613,0.425095,0.397676,0.340644
Allegany,0.269162,NaN,0.231897,0.355196,0.292518,0.402621,0.804433,0.840649,0.670524,0.521700,...,0.214244,0.308360,0.493541,0.366303,0.375316,0.808667,0.830615,0.328521,0.281157,0.231445
Anne Arundel,0.347338,0.342971,0.336577,0.450729,0.446952,0.473227,0.645089,0.818086,0.826633,0.426275,...,0.490064,0.477448,0.349229,0.338087,0.565987,0.814249,0.779990,0.549313,0.323787,0.341968



✅ Success! The combined soybean data has been saved to: '/content/drive/MyDrive/GEE_Exports/md_soybean_ndvi_timeseries_combined_wide.csv'


In [52]:
# Add this to a new cell and run it
print("The shape of the final table is (rows, columns):")
print(wide_df.shape)

The shape of the final table is (rows, columns):
(50, 47)


In [60]:
# --- 1. DEFINE THE MASTER LIST OF MARYLAND COUNTIES ---
# This list contains the 23 counties plus Baltimore City.
maryland_counties_list = [
    'Allegany', 'Anne Arundel', 'Baltimore', 'Baltimore City', 'Calvert',
    'Caroline', 'Carroll', 'Cecil', 'Charles', 'Dorchester', 'Frederick',
    'Garrett', 'Harford', 'Howard', 'Kent', 'Montgomery', "Prince George's",
    "Queen Anne's", 'Somerset', "St. Mary's", 'Talbot', 'Washington',
    'Wicomico', 'Worcester'
]

In [61]:
# --- 2. DEFINE FILE PATHS ---
# Make sure this is the correct path to your combined file in Google Drive.
input_file = '/content/drive/My Drive/GEE_Exports/md_soybean_ndvi_timeseries_combined_wide.csv'
output_file = '/content/drive/My Drive/GEE_Exports/maryland_only_soybean_ndvi_timeseries_FINAL.csv'

In [62]:
# --- 3. LOAD THE WIDE-FORMAT DATA ---
# We tell pandas that the first column ('NAME') should be the index.
try:
    wide_df = pd.read_csv(input_file, index_col='NAME')
    print(f"Successfully loaded the combined data. Original shape: {wide_df.shape}")
except FileNotFoundError:
    print(f"Error: The input file was not found at: {input_file}")
    wide_df = None

Successfully loaded the combined data. Original shape: (50, 47)


In [64]:
# --- 4. FILTER THE DATAFRAME ---
if wide_df is not None:
    # The .isin() function checks each county in the table's index
    # and keeps only those that are in our maryland_counties_list.
    final_df = wide_df[wide_df.index.isin(maryland_counties_list)]

    print(f"Filtered down to Maryland counties. Final shape: {final_df.shape}")

    # --- 5. SAVE THE FINAL, CLEAN CSV ---
    final_df.to_csv(output_file)
    print(f"\n✅ Success! The final, clean dataset has been saved to: '{output_file}'")

    # Display a preview of the final, clean table.
    print("\nPreview of the final, Maryland-only data:")
    display(final_df.head())

Filtered down to Maryland counties. Final shape: (23, 47)

✅ Success! The final, clean dataset has been saved to: '/content/drive/My Drive/GEE_Exports/maryland_only_soybean_ndvi_timeseries_FINAL.csv'

Preview of the final, Maryland-only data:


,SoybeanNDVI_2021-01,SoybeanNDVI_2021-02,SoybeanNDVI_2021-03,SoybeanNDVI_2021-04,SoybeanNDVI_2021-05,SoybeanNDVI_2021-06,SoybeanNDVI_2021-07,SoybeanNDVI_2021-08,SoybeanNDVI_2021-09,SoybeanNDVI_2021-10,...,SoybeanNDVI_2024-03,SoybeanNDVI_2024-04,SoybeanNDVI_2024-05,SoybeanNDVI_2024-06,SoybeanNDVI_2024-07,SoybeanNDVI_2024-08,SoybeanNDVI_2024-09,SoybeanNDVI_2024-10,SoybeanNDVI_2024-11,SoybeanNDVI_2024-12
NAME,,,,,,,,,,,,,,,,,,,,,
Allegany,0.269162,NaN,0.231897,0.355196,0.292518,0.402621,0.804433,0.840649,0.670524,0.521700,...,0.214244,0.308360,0.493541,0.366303,0.375316,0.808667,0.830615,0.328521,0.281157,0.231445
Anne Arundel,0.347338,0.342971,0.336577,0.450729,0.446952,0.473227,0.645089,0.818086,0.826633,0.426275,...,0.490064,0.477448,0.349229,0.338087,0.565987,0.814249,0.779990,0.549313,0.323787,0.341968
Baltimore,0.371915,0.330855,0.324762,0.425404,0.478987,0.459140,0.528149,0.625620,0.688803,0.435925,...,0.553673,0.632860,0.417793,0.399544,0.534607,0.839457,0.799097,0.443454,0.460811,0.446721
Calvert,0.371684,0.342152,0.340685,0.510428,0.501592,0.472225,0.567817,0.793863,0.805532,0.500274,...,0.525548,0.485066,0.382534,0.334999,0.473227,0.757385,0.761279,0.525778,0.387645,0.385753
Caroline,0.379205,0.366621,0.377498,0.490088,0.447504,0.460139,0.615064,0.818829,0.836925,0.455907,...,0.546389,0.534476,0.432372,0.342659,0.582751,0.848254,0.811261,0.446820,0.282559,0.340688
